# Analysis

## Phase-1: Churn Labeling (BG/NBD)

BG/NBD (Beta Geometric / Negative Binomial Distribution) model is used to assign churn labels.

**Calibration period:** subreddit creation to July 1, 2025

**Holdout period:** July 1, 2025 to January 1, 2026 (6 months)

A user is classified as **churned** if:
- `p_alive < 0.10` at the end of the observation window 
- AND they made no posts during the holdout period. 

Otherwise they are classified as **active**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROCESSED_DIR = '../data/processed_data/'

# activity records
activity_lp     = pd.read_parquet(PROCESSED_DIR + 'activity_lp.parquet')
activity_loseit = pd.read_parquet(PROCESSED_DIR + 'activity_loseit.parquet')
activity_dep    = pd.read_parquet(PROCESSED_DIR + 'activity_dep.parquet')

# cleaned posts and comments
posts_lp        = pd.read_parquet(PROCESSED_DIR + 'posts_lp.parquet')
posts_loseit    = pd.read_parquet(PROCESSED_DIR + 'posts_loseit.parquet')
posts_dep       = pd.read_parquet(PROCESSED_DIR + 'posts_dep.parquet')

comments_lp     = pd.read_parquet(PROCESSED_DIR + 'comments_lp.parquet')
comments_loseit = pd.read_parquet(PROCESSED_DIR + 'comments_loseit.parquet')
comments_dep    = pd.read_parquet(PROCESSED_DIR + 'comments_dep.parquet')

# user stats
user_stats_lp     = pd.read_parquet(PROCESSED_DIR + 'user_stats_lp.parquet')
user_stats_loseit = pd.read_parquet(PROCESSED_DIR + 'user_stats_loseit.parquet')
user_stats_dep    = pd.read_parquet(PROCESSED_DIR + 'user_stats_dep.parquet')

# reply edge lists
edges_lp        = pd.read_parquet(PROCESSED_DIR + 'edges_lp.parquet')
edges_loseit    = pd.read_parquet(PROCESSED_DIR + 'edges_loseit.parquet')
edges_dep       = pd.read_parquet(PROCESSED_DIR + 'edges_dep.parquet')

print("all DataFrames loaded")

In [1]:
from lifetimes import BetaGeoFitter

In [4]:
# calibration / holdout split
HOLDOUT_START   = pd.Timestamp('2025-07-01', tz='UTC')
OBS_END         = pd.Timestamp('2026-01-01', tz='UTC')
T_holdout_weeks = (OBS_END - HOLDOUT_START).days / 7

CHURN_THRESHOLD = 0.10

print(f"calibration ends:  {HOLDOUT_START.date()}")
print(f"holdout period:    {HOLDOUT_START.date()} to {OBS_END.date()}  ({T_holdout_weeks:.1f} weeks)")
print(f"churn threshold:   p_alive < {CHURN_THRESHOLD}")

calibration ends:  2025-07-01
holdout period:    2025-07-01 to 2026-01-01  (26.3 weeks)
churn threshold:   p_alive < 0.1


### Subreddit-1: learnprogramming

In [5]:
# split activity into calibration and holdout periods
cal_lp = activity_lp[activity_lp['created_utc'] < HOLDOUT_START].copy()
hld_lp = activity_lp[activity_lp['created_utc'] >= HOLDOUT_START].copy()

print(f"calibration records: {len(cal_lp):,}")
print(f"holdout records:     {len(hld_lp):,}")

NameError: name 'activity_lp' is not defined

In [ ]:
# compute BG/NBD inputs from calibration period
bgnbd_cal_lp = cal_lp.groupby('author').agg(
    n_cal      = ('created_utc', 'count'),
    first_post = ('created_utc', 'min'),
    last_cal   = ('created_utc', 'max')
).reset_index()

bgnbd_cal_lp['T_cal']         = (HOLDOUT_START - bgnbd_cal_lp['first_post']).dt.days / 7
bgnbd_cal_lp['recency_cal']   = (bgnbd_cal_lp['last_cal'] - bgnbd_cal_lp['first_post']).dt.days / 7
bgnbd_cal_lp['frequency_cal'] = bgnbd_cal_lp['n_cal'] - 1

bgnbd_cal_lp = bgnbd_cal_lp[bgnbd_cal_lp['T_cal'] > 0].reset_index(drop=True)

print(f"calibration users: {len(bgnbd_cal_lp):,}")
bgnbd_cal_lp[['frequency_cal', 'recency_cal', 'T_cal']].describe().round(2)

In [ ]:
# check who was active during holdout
holdout_users_lp = set(hld_lp['author'].unique())
bgnbd_cal_lp['active_holdout'] = bgnbd_cal_lp['author'].isin(holdout_users_lp).astype(int)

print(f"active in holdout:   {bgnbd_cal_lp['active_holdout'].sum():,}  ({bgnbd_cal_lp['active_holdout'].mean()*100:.1f}%)")
print(f"inactive in holdout: {(bgnbd_cal_lp['active_holdout']==0).sum():,}  ({(1-bgnbd_cal_lp['active_holdout'].mean())*100:.1f}%)")

In [ ]:
# fit BG/NBD model on calibration data
bgf_lp = BetaGeoFitter(penalizer_coef=0.001)
bgf_lp.fit(
    frequency = bgnbd_cal_lp['frequency_cal'],
    recency   = bgnbd_cal_lp['recency_cal'],
    T         = bgnbd_cal_lp['T_cal']
)
print(bgf_lp)

In [ ]:
# compute p_alive at OBS_END (extend T by holdout period length)
bgnbd_cal_lp['p_alive'] = bgf_lp.conditional_probability_alive(
    frequency = bgnbd_cal_lp['frequency_cal'],
    recency   = bgnbd_cal_lp['recency_cal'],
    T         = bgnbd_cal_lp['T_cal'] + T_holdout_weeks
)

bgnbd_cal_lp['p_alive'].describe().round(4)

In [ ]:
# p_alive distribution
plt.figure(figsize=(10, 4))
plt.hist(bgnbd_cal_lp['p_alive'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(x=CHURN_THRESHOLD, color='red', linestyle='--', label=f'churn threshold = {CHURN_THRESHOLD}')
plt.title('P(alive) Distribution — r/learnprogramming', fontweight='bold')
plt.xlabel('P(alive)')
plt.ylabel('Number of Users')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# assign churn label
# churned = p_alive < threshold AND no activity in holdout
bgnbd_cal_lp['churned'] = (
    (bgnbd_cal_lp['p_alive'] < CHURN_THRESHOLD) &
    (bgnbd_cal_lp['active_holdout'] == 0)
).astype(int)

n_total_lp   = len(bgnbd_cal_lp)
n_churned_lp = bgnbd_cal_lp['churned'].sum()

print(f"r/learnprogramming — churn labels")
print(f"  total users: {n_total_lp:,}")
print(f"  churned:     {n_churned_lp:,}  ({n_churned_lp/n_total_lp*100:.1f}%)")
print(f"  active:      {n_total_lp - n_churned_lp:,}  ({(n_total_lp - n_churned_lp)/n_total_lp*100:.1f}%)")

### Subreddit-2: loseit

In [ ]:
# split activity into calibration and holdout periods
cal_loseit = activity_loseit[activity_loseit['created_utc'] < HOLDOUT_START].copy()
hld_loseit = activity_loseit[activity_loseit['created_utc'] >= HOLDOUT_START].copy()

print(f"calibration records: {len(cal_loseit):,}")
print(f"holdout records:     {len(hld_loseit):,}")

In [ ]:
# compute BG/NBD inputs from calibration period
bgnbd_cal_loseit = cal_loseit.groupby('author').agg(
    n_cal      = ('created_utc', 'count'),
    first_post = ('created_utc', 'min'),
    last_cal   = ('created_utc', 'max')
).reset_index()

bgnbd_cal_loseit['T_cal']         = (HOLDOUT_START - bgnbd_cal_loseit['first_post']).dt.days / 7
bgnbd_cal_loseit['recency_cal']   = (bgnbd_cal_loseit['last_cal'] - bgnbd_cal_loseit['first_post']).dt.days / 7
bgnbd_cal_loseit['frequency_cal'] = bgnbd_cal_loseit['n_cal'] - 1

bgnbd_cal_loseit = bgnbd_cal_loseit[bgnbd_cal_loseit['T_cal'] > 0].reset_index(drop=True)

print(f"calibration users: {len(bgnbd_cal_loseit):,}")
bgnbd_cal_loseit[['frequency_cal', 'recency_cal', 'T_cal']].describe().round(2)

In [ ]:
# check who was active during holdout
holdout_users_loseit = set(hld_loseit['author'].unique())
bgnbd_cal_loseit['active_holdout'] = bgnbd_cal_loseit['author'].isin(holdout_users_loseit).astype(int)

print(f"active in holdout:   {bgnbd_cal_loseit['active_holdout'].sum():,}  ({bgnbd_cal_loseit['active_holdout'].mean()*100:.1f}%)")
print(f"inactive in holdout: {(bgnbd_cal_loseit['active_holdout']==0).sum():,}  ({(1-bgnbd_cal_loseit['active_holdout'].mean())*100:.1f}%)")

In [ ]:
# fit BG/NBD model
bgf_loseit = BetaGeoFitter(penalizer_coef=0.001)
bgf_loseit.fit(
    frequency = bgnbd_cal_loseit['frequency_cal'],
    recency   = bgnbd_cal_loseit['recency_cal'],
    T         = bgnbd_cal_loseit['T_cal']
)
print(bgf_loseit)

In [ ]:
# compute p_alive at OBS_END
bgnbd_cal_loseit['p_alive'] = bgf_loseit.conditional_probability_alive(
    frequency = bgnbd_cal_loseit['frequency_cal'],
    recency   = bgnbd_cal_loseit['recency_cal'],
    T         = bgnbd_cal_loseit['T_cal'] + T_holdout_weeks
)

bgnbd_cal_loseit['p_alive'].describe().round(4)

In [ ]:
# p_alive distribution
plt.figure(figsize=(10, 4))
plt.hist(bgnbd_cal_loseit['p_alive'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(x=CHURN_THRESHOLD, color='red', linestyle='--', label=f'churn threshold = {CHURN_THRESHOLD}')
plt.title('P(alive) Distribution — r/loseit', fontweight='bold')
plt.xlabel('P(alive)')
plt.ylabel('Number of Users')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# assign churn label
bgnbd_cal_loseit['churned'] = (
    (bgnbd_cal_loseit['p_alive'] < CHURN_THRESHOLD) &
    (bgnbd_cal_loseit['active_holdout'] == 0)
).astype(int)

n_total_loseit   = len(bgnbd_cal_loseit)
n_churned_loseit = bgnbd_cal_loseit['churned'].sum()

print(f"r/loseit — churn labels")
print(f"  total users: {n_total_loseit:,}")
print(f"  churned:     {n_churned_loseit:,}  ({n_churned_loseit/n_total_loseit*100:.1f}%)")
print(f"  active:      {n_total_loseit - n_churned_loseit:,}  ({(n_total_loseit - n_churned_loseit)/n_total_loseit*100:.1f}%)")

### Subreddit-3: depression

In [ ]:
# split activity into calibration and holdout periods
cal_dep = activity_dep[activity_dep['created_utc'] < HOLDOUT_START].copy()
hld_dep = activity_dep[activity_dep['created_utc'] >= HOLDOUT_START].copy()

print(f"calibration records: {len(cal_dep):,}")
print(f"holdout records:     {len(hld_dep):,}")

In [ ]:
# compute BG/NBD inputs from calibration period
bgnbd_cal_dep = cal_dep.groupby('author').agg(
    n_cal      = ('created_utc', 'count'),
    first_post = ('created_utc', 'min'),
    last_cal   = ('created_utc', 'max')
).reset_index()

bgnbd_cal_dep['T_cal']         = (HOLDOUT_START - bgnbd_cal_dep['first_post']).dt.days / 7
bgnbd_cal_dep['recency_cal']   = (bgnbd_cal_dep['last_cal'] - bgnbd_cal_dep['first_post']).dt.days / 7
bgnbd_cal_dep['frequency_cal'] = bgnbd_cal_dep['n_cal'] - 1

bgnbd_cal_dep = bgnbd_cal_dep[bgnbd_cal_dep['T_cal'] > 0].reset_index(drop=True)

print(f"calibration users: {len(bgnbd_cal_dep):,}")
bgnbd_cal_dep[['frequency_cal', 'recency_cal', 'T_cal']].describe().round(2)

In [ ]:
# check who was active during holdout
holdout_users_dep = set(hld_dep['author'].unique())
bgnbd_cal_dep['active_holdout'] = bgnbd_cal_dep['author'].isin(holdout_users_dep).astype(int)

print(f"active in holdout:   {bgnbd_cal_dep['active_holdout'].sum():,}  ({bgnbd_cal_dep['active_holdout'].mean()*100:.1f}%)")
print(f"inactive in holdout: {(bgnbd_cal_dep['active_holdout']==0).sum():,}  ({(1-bgnbd_cal_dep['active_holdout'].mean())*100:.1f}%)")

In [ ]:
# fit BG/NBD model
bgf_dep = BetaGeoFitter(penalizer_coef=0.001)
bgf_dep.fit(
    frequency = bgnbd_cal_dep['frequency_cal'],
    recency   = bgnbd_cal_dep['recency_cal'],
    T         = bgnbd_cal_dep['T_cal']
)
print(bgf_dep)

In [ ]:
# compute p_alive at OBS_END
bgnbd_cal_dep['p_alive'] = bgf_dep.conditional_probability_alive(
    frequency = bgnbd_cal_dep['frequency_cal'],
    recency   = bgnbd_cal_dep['recency_cal'],
    T         = bgnbd_cal_dep['T_cal'] + T_holdout_weeks
)

bgnbd_cal_dep['p_alive'].describe().round(4)

In [ ]:
# p_alive distribution
plt.figure(figsize=(10, 4))
plt.hist(bgnbd_cal_dep['p_alive'], bins=50, color='steelblue', edgecolor='white')
plt.axvline(x=CHURN_THRESHOLD, color='red', linestyle='--', label=f'churn threshold = {CHURN_THRESHOLD}')
plt.title('P(alive) Distribution — r/depression', fontweight='bold')
plt.xlabel('P(alive)')
plt.ylabel('Number of Users')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# assign churn label
bgnbd_cal_dep['churned'] = (
    (bgnbd_cal_dep['p_alive'] < CHURN_THRESHOLD) &
    (bgnbd_cal_dep['active_holdout'] == 0)
).astype(int)

n_total_dep   = len(bgnbd_cal_dep)
n_churned_dep = bgnbd_cal_dep['churned'].sum()

print(f"r/depression — churn labels")
print(f"  total users: {n_total_dep:,}")
print(f"  churned:     {n_churned_dep:,}  ({n_churned_dep/n_total_dep*100:.1f}%)")
print(f"  active:      {n_total_dep - n_churned_dep:,}  ({(n_total_dep - n_churned_dep)/n_total_dep*100:.1f}%)")

### Summary

In [ ]:
# churn label summary across all three subreddits
summary = pd.DataFrame({
    'Subreddit':     ['r/learnprogramming', 'r/loseit', 'r/depression'],
    'Total users':   [n_total_lp,   n_total_loseit,   n_total_dep],
    'Churned':       [n_churned_lp, n_churned_loseit, n_churned_dep],
    'Active':        [n_total_lp - n_churned_lp,
                      n_total_loseit - n_churned_loseit,
                      n_total_dep - n_churned_dep],
    'Churn rate (%)': [round(n_churned_lp/n_total_lp*100, 1),
                       round(n_churned_loseit/n_total_loseit*100, 1),
                       round(n_churned_dep/n_total_dep*100, 1)]
})
summary